In [1]:
import pandas as pd

In [2]:
file_path = "multiqc_data/multiqc_fastqc.txt"
df = pd.read_csv(file_path, sep="\t")

print(df.head())  # View first few rows


   Sample      Filename                File type      Encoding  \
0  AZ_A10  AZ_A10.fastq  Conventional base calls  Illumina 1.5   
1  AZ_A11  AZ_A11.fastq  Conventional base calls  Illumina 1.5   
2   AZ_A6   AZ_A6.fastq  Conventional base calls  Illumina 1.5   
3   AZ_A7   AZ_A7.fastq  Conventional base calls  Illumina 1.5   
4   AZ_A8   AZ_A8.fastq  Conventional base calls  Illumina 1.5   

   Total Sequences Total Bases  Sequences flagged as poor quality  \
0        1430710.0    61.5 Mbp                                0.0   
1        1105884.0    47.5 Mbp                                0.0   
2        1276330.0    54.8 Mbp                                0.0   
3        3386434.0   145.6 Mbp                                0.0   
4         880487.0    37.8 Mbp                                0.0   

   Sequence length   %GC  total_deduplicated_percentage  ...  \
0             43.0  47.0                      62.557450  ...   
1             43.0  45.0                      62.112437  ...

In [4]:
df.columns

Index(['Sample', 'Filename', 'File type', 'Encoding', 'Total Sequences',
       'Total Bases', 'Sequences flagged as poor quality', 'Sequence length',
       '%GC', 'total_deduplicated_percentage', 'avg_sequence_length',
       'median_sequence_length', 'basic_statistics',
       'per_base_sequence_quality', 'per_sequence_quality_scores',
       'per_base_sequence_content', 'per_sequence_gc_content',
       'per_base_n_content', 'sequence_length_distribution',
       'sequence_duplication_levels', 'overrepresented_sequences',
       'adapter_content'],
      dtype='object')

In [8]:
columns_to_keep = [
    'Sample', 'Total Sequences', 'Sequences flagged as poor quality', '%GC',
    'per_base_sequence_quality', 'per_sequence_gc_content', 'per_base_n_content',
    'adapter_content', 'sequence_duplication_levels'
]

df_filtered = df[columns_to_keep]
df_filtered.head()

,Sample,Total Sequences,Sequences flagged as poor quality,%GC,per_base_sequence_quality,per_sequence_gc_content,per_base_n_content,adapter_content,sequence_duplication_levels
0,AZ_A10,1430710.0,0.0,47.0,pass,warn,pass,pass,warn
1,AZ_A11,1105884.0,0.0,45.0,pass,fail,pass,pass,warn
2,AZ_A6,1276330.0,0.0,43.0,pass,pass,pass,pass,warn
3,AZ_A7,3386434.0,0.0,48.0,pass,pass,pass,pass,fail
4,AZ_A8,880487.0,0.0,47.0,pass,warn,pass,pass,warn


In [9]:
df_filtered = df_filtered.rename(columns={
    'Total Sequences': 'Total_Reads',
    'Sequences flagged as poor quality': 'Poor_Q_Reads',
    'per_base_sequence_quality': 'Avg_Q',
    'per_sequence_gc_content': 'GC_Content',
    'per_base_n_content': 'N_Content',
    'sequence_duplication_levels': 'Duplication_Level'
})
df_filtered.head()


,Sample,Total_Reads,Poor_Q_Reads,%GC,Avg_Q,GC_Content,N_Content,adapter_content,Duplication_Level
0,AZ_A10,1430710.0,0.0,47.0,pass,warn,pass,pass,warn
1,AZ_A11,1105884.0,0.0,45.0,pass,fail,pass,pass,warn
2,AZ_A6,1276330.0,0.0,43.0,pass,pass,pass,pass,warn
3,AZ_A7,3386434.0,0.0,48.0,pass,pass,pass,pass,fail
4,AZ_A8,880487.0,0.0,47.0,pass,warn,pass,pass,warn


In [10]:
df_filtered.shape

(3274, 9)

In [ ]:
low_quality_threshold = 20
high_n_content = 5  # %N content
high_adapter_content = 10  # Adapter contamination

df['Low_Quality'] = df['per_base_sequence_quality'] < low_quality_threshold
df['High_N_Content'] = df['Per base N content'] > high_n_content
df['High_Adapter_Content'] = df['Adapter Content'] > high_adapter_content

flagged_samples = df[(df['Low_Quality']) | (df['High_N_Content']) | (df['High_Adapter_Content'])]
print("FASTQ files to consider removing:")
print(flagged_samples[['Sample', 'Low_Quality', 'High_N_Content', 'High_Adapter_Content']])
